# Geometric defects of midpoint projection — saved-data figures

Load the same record as `01_compute.ipynb` and create the French-labelled figures for thesis section 4.9. This notebook does not integrate any trajectories. It reads the complete local study and calls the shared thesis plotting script. If the recovery cache is missing, run the calculation notebook with its default `RUN_COMPUTATION=False` to download completed results.

The local defects compare identical physical points and forcing phases. The accumulated log determinant measures area change along each computed trajectory; a long-time return displacement also includes perturbation amplification. Do not infer exact structure or asymptotic order from values at a numerical floor.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/studies/midpoint_geometry.py").is_file())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

STUDY_PREFIX = "studies/midpoint_geometry"
RUN_ID = "midpoint-geometry-20261118-rho0015"
RUN_DIRECTORY = ROOT / "outputs/midpoint_geometry" / RUN_ID
BUCKET = f"gc2d_data:gc2d-notebooks-data/{STUDY_PREFIX}/{RUN_ID}"
FILES = ("local.npz", "seams.npz", "fd_audit.npz", "calibration.npz", "implicit_controls.npz",
         "accumulation.npz", "summary.json", "manifest.json", "source.tar.gz")


In [ ]:
missing = [name for name in FILES if not (RUN_DIRECTORY / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing saved study files: {missing}. Recover them with 01_compute.ipynb without enabling computation.")
summary = json.loads((RUN_DIRECTORY / "summary.json").read_text())
print(json.dumps(summary, indent=2))
with np.load(RUN_DIRECTORY / "local.npz", allow_pickle=False) as local:
    assert local["det_defect"].shape == (2, 4, 4, 84)
    assert local["return_norm"].shape == local["det_defect"].shape


## Export thesis figures

The plotting script loads the numerical archives and writes vector PDF figures, PNG companions and JSON provenance. Labels, legends and annotations are in French. No thesis PDF is opened or rendered. Figure revisions reuse these scientific data.

In [ ]:
import subprocess

THESIS_ROOT = ROOT.parent / "TFM_redaccion"
PLOT_SCRIPT = THESIS_ROOT / "notebooks/geometria_midpoint_graficas.py"
if not PLOT_SCRIPT.is_file():
    raise FileNotFoundError(f"The shared thesis plotting script is unavailable: {PLOT_SCRIPT}")
completed = subprocess.run(
    [sys.executable, str(PLOT_SCRIPT), "--results", str(RUN_DIRECTORY.resolve())],
    cwd=THESIS_ROOT, check=True, text=True, capture_output=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
